# C1 — BVP + EDA Preprocessing (WESAD)

Standalone preprocessing extracted from `c1_bvp_eda_full_pipeline.ipynb`. It loads WESAD wrist BVP and EDA, cleans and aligns the signals, extracts window features, exports the dataset, removes duplicate features, and optionally performs per-subject normalization.

The source full-pipeline notebook remains unchanged.


---
# 0 · Setup


In [ ]:
# ── Colab only ────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not running in Colab — set paths manually in the config cell.')

In [ ]:
# !pip install -q scipy pandas matplotlib seaborn
print('Preprocessing dependencies ready.')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION — edit this cell only
# ══════════════════════════════════════════════════════════════════════════════

# Folder containing WESAD subject directories such as S2/, S3/, ...
WESAD_PATH = '/content/drive/MyDrive/Data_Set/WESAD/WESAD'

# Output directory for the cached preprocessed datasets.
OUT_ROOT = '/content/drive/MyDrive/C1_BVP_EDA'

# Native WESAD sampling rates.
BVP_FS = 64
EDA_FS = 4
LABEL_FS = 700
TARGET_FS = 32

# Windowing.
WINDOW_SEC = 60.0
STEP_SEC = 5.0
LABEL_PURITY = 0.95

# WESAD subjects and conditions. S12 is excluded from the public dataset.
SUBJECTS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
LABEL_MAP = {1: 0, 2: 1}  # baseline -> 0, stress -> 1
RANDOM_SEED = 42

# Set False to load an existing dataset_bvp_eda.npz instead of rebuilding it.
RUN_PREPROCESS = True

# Optional per-subject/device normalization using the first five minutes.
PER_SUBJECT_NORM = True
CALIB_MODE = 'first_k'  # 'first_k' | 'baseline_label' | 'none'
CALIB_SEC = 300.0
CALIB_WINDOWS = max(1, int((CALIB_SEC - WINDOW_SEC) // STEP_SEC) + 1)
DROP_DUPLICATE_FEATURES = True
DUPLICATE_FEATURES = ['bvp_std_ibi', 'eda_tonic_mean']

import os
RUN_TAG = f"bvpeda_{'persubj_' + CALIB_MODE if PER_SUBJECT_NORM else 'global'}_w{int(WINDOW_SEC)}s"
OUT_PATH = os.path.join(OUT_ROOT, RUN_TAG)
os.makedirs(OUT_PATH, exist_ok=True)

WINDOW_LEN = int(WINDOW_SEC * TARGET_FS)
STEP_LEN = int(STEP_SEC * TARGET_FS)
OVERLAP_PCT = max(0.0, 100.0 * (WINDOW_SEC - STEP_SEC) / WINDOW_SEC)

print(f'Native rates: BVP={BVP_FS} Hz | EDA={EDA_FS} Hz | labels={LABEL_FS} Hz')
print(f'Window: {WINDOW_SEC}s @ {TARGET_FS} Hz = {WINDOW_LEN} timesteps')
print(f'Step: {STEP_SEC}s ({OVERLAP_PCT:.1f}% overlap)')
print(f'Calibration windows: {CALIB_WINDOWS} (~{CALIB_SEC / 60:.1f} min)')
print(f'Output: {OUT_PATH}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  IMPORTS & REPRODUCIBILITY
# ══════════════════════════════════════════════════════════════════════════════
import os
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import butter, filtfilt, find_peaks, resample_poly

sns.set_theme(style='whitegrid', font_scale=1.05)
np.random.seed(RANDOM_SEED)
print('Imports ready.')


---
# PART A · Preprocessing

Two synchronized representations are built in one pass:

| Output | Shape | Consumers |
|---|---|---|
| `X_feat` | `(N, 20)` before duplicate removal | LR, RF, SVM, HGB, Feature-MLP |
| `X_raw`  | `(N, WINDOW_LEN, 2)` | optional CNN-LSTM |
| `y`      | `(N,)` | all models |
| `groups` | `(N,)` | subject ID for subject-wise validation |

**Important change from the ECG notebook:** pulse peaks are detected on the native **64 Hz BVP**
signal. The intervals between consecutive optical pulse peaks are called **IBI / pulse intervals**.
The derived variability measures are treated as **PRV** (pulse-rate variability), not ECG HRV.

Wrist EDA stays at its native **4 Hz** for EDA feature extraction. For the optional raw CNN-LSTM,
BVP and EDA are resampled to a common 32 Hz timeline and stacked as `[EDA, BVP]`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  SIGNAL PROCESSING HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def butter_filter(x, fs, cutoff, btype='low', order=4):
    """Zero-phase Butterworth filter (filtfilt -> no phase distortion)."""
    x = np.asarray(x, dtype=np.float64).ravel()
    nyq = 0.5 * fs
    if btype == 'band':
        wn = [cutoff[0] / nyq, cutoff[1] / nyq]
    else:
        wn = cutoff / nyq
    wn = np.clip(wn, 1e-6, 0.999)
    b, a = butter(order, wn, btype=btype)
    return filtfilt(b, a, x)


def detect_bvp_peaks(bvp, fs):
    """
    Detect optical pulse peaks from wrist BVP.

    Steps:
      1) band-pass 0.5-4 Hz (roughly 30-240 cycles/min)
      2) robustly standardize by median/MAD
      3) adaptive prominence peak detection

    Returns peak sample indices at the ORIGINAL BVP sampling rate.
    """
    filt = butter_filter(bvp, fs, [0.5, 4.0], btype='band', order=3)

    med = np.median(filt)
    mad = np.median(np.abs(filt - med))
    robust_sd = 1.4826 * mad
    if not np.isfinite(robust_sd) or robust_sd < 1e-8:
        robust_sd = np.std(filt) + 1e-8
    z = (filt - med) / robust_sd

    peaks, _ = find_peaks(
        z,
        distance=max(1, int(0.30 * fs)),   # <=200 bpm
        prominence=0.5,
    )
    return peaks


def decompose_eda(eda, fs):
    """
    Simple wrist-EDA decomposition.
    E4 EDA is 4 Hz, so the cleaning low-pass must stay below its 2 Hz Nyquist.
    """
    eda = np.asarray(eda, dtype=np.float64).ravel()
    eda_clean = butter_filter(eda, fs, 1.0, btype='low', order=4)
    tonic     = butter_filter(eda_clean, fs, 0.05, btype='low', order=2)
    phasic    = eda_clean - tonic
    return eda_clean, tonic, phasic


print('BVP + EDA signal helpers ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FEATURE EXTRACTION — 20 features per window before duplicate removal
# ══════════════════════════════════════════════════════════════════════════════

FEATURE_COLS = [
    # ── EDA (12) ──
    'eda_mean', 'eda_std', 'eda_min', 'eda_max', 'eda_range', 'eda_slope',
    'eda_num_peaks', 'eda_peak_amplitude',
    'eda_phasic_mean', 'eda_phasic_std', 'eda_phasic_max', 'eda_tonic_mean',

    # ── BVP / pulse timing / PRV (8) ──
    'bvp_mean_ibi', 'bvp_std_ibi', 'bvp_min_ibi', 'bvp_max_ibi',
    'bvp_heart_rate', 'bvp_prv_sdnn', 'bvp_prv_rmssd', 'bvp_prv_pnn50',
]

COLLINEAR_COLS = ['bvp_heart_rate', 'eda_range']


def eda_window_features(eda_w, tonic_w, phasic_w, fs):
    """12 EDA features for one window."""
    eda_w    = np.asarray(eda_w, dtype=np.float64)
    tonic_w  = np.asarray(tonic_w, dtype=np.float64)
    phasic_w = np.asarray(phasic_w, dtype=np.float64)

    if len(eda_w) < 3:
        return [np.nan] * 12

    t = np.arange(len(eda_w)) / fs
    slope = np.polyfit(t, eda_w, 1)[0]

    # SCR-like peaks in the phasic component.
    pk, props = find_peaks(
        phasic_w,
        prominence=0.01,
        distance=max(1, int(1.0 * fs)),
    )
    peak_amp = float(np.mean(props['prominences'])) if len(pk) else 0.0

    return [
        float(np.mean(eda_w)), float(np.std(eda_w)),
        float(np.min(eda_w)),  float(np.max(eda_w)),
        float(np.max(eda_w) - np.min(eda_w)),
        float(slope),
        float(len(pk)), peak_amp,
        float(np.mean(phasic_w)), float(np.std(phasic_w)), float(np.max(phasic_w)),
        float(np.mean(tonic_w)),
    ]


def pulse_interval_features(ibi_ms):
    """
    8 pulse-timing features from BVP inter-beat intervals (IBI, ms).

    SDNN/RMSSD/pNN50 derived from optical pulse intervals are PRV features.
    They are analogous to ECG-derived HRV features but are not identical measurements.
    """
    ibi_ms = np.asarray(ibi_ms, dtype=np.float64)
    ibi_ms = ibi_ms[np.isfinite(ibi_ms)]

    if len(ibi_ms) < 2:
        return [np.nan] * 8

    mean_ibi = float(np.mean(ibi_ms))
    diff_ibi = np.diff(ibi_ms)

    return [
        mean_ibi,
        float(np.std(ibi_ms)),
        float(np.min(ibi_ms)),
        float(np.max(ibi_ms)),
        60000.0 / mean_ibi,
        float(np.std(ibi_ms, ddof=1)),
        float(np.sqrt(np.mean(diff_ibi ** 2))) if len(diff_ibi) else np.nan,
        float(np.mean(np.abs(diff_ibi) > 50) * 100.0) if len(diff_ibi) else np.nan,
    ]


print(f'{len(FEATURE_COLS)} BVP/PRV + EDA features defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PER-SUBJECT PROCESSING — WESAD WRIST BVP + WRIST EDA
# ══════════════════════════════════════════════════════════════════════════════

def process_subject(sid, wesad_path):
    """
    Load one WESAD subject from S{sid}.pkl and return synchronized:
      features : (n_windows, 20)
      raw      : (n_windows, WINDOW_LEN, 2)  channels = [EDA, BVP]
      labels   : (n_windows,)
      groups   : subject id for each window

    Native signal rates are different (BVP=64 Hz, EDA=4 Hz, labels=700 Hz),
    so all slicing is done by TIME in seconds rather than by matching array index.
    """
    pkl = os.path.join(wesad_path, f'S{sid}', f'S{sid}.pkl')
    with open(pkl, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    # ── 1. Load the WRIST channels ───────────────────────────────────────────
    wrist   = data['signal']['wrist']
    bvp_raw = np.asarray(wrist['BVP'], dtype=np.float64).ravel()
    eda_raw = np.asarray(wrist['EDA'], dtype=np.float64).ravel()
    labels  = np.asarray(data['label']).ravel()

    # Restrict all streams to their common time duration.
    duration = min(len(bvp_raw) / BVP_FS,
                   len(eda_raw) / EDA_FS,
                   len(labels)  / LABEL_FS)

    bvp_raw = bvp_raw[:int(np.floor(duration * BVP_FS))]
    eda_raw = eda_raw[:int(np.floor(duration * EDA_FS))]
    labels  = labels [:int(np.floor(duration * LABEL_FS))]
    duration = min(len(bvp_raw) / BVP_FS,
                   len(eda_raw) / EDA_FS,
                   len(labels)  / LABEL_FS)

    # ── 2. Pulse peaks / IBI at native 64 Hz ─────────────────────────────────
    pulse_peaks = detect_bvp_peaks(bvp_raw, BVP_FS)
    pulse_t     = pulse_peaks / BVP_FS
    ibi_all     = np.diff(pulse_t) * 1000.0
    ibi_mid_t   = (pulse_t[:-1] + pulse_t[1:]) / 2.0

    # Physiological plausibility gate: 30-200 bpm.
    ok = (ibi_all > 300.0) & (ibi_all < 2000.0)
    ibi_all, ibi_mid_t = ibi_all[ok], ibi_mid_t[ok]

    # ── 3. EDA decomposition at native 4 Hz ──────────────────────────────────
    eda_clean, tonic, phasic = decompose_eda(eda_raw, EDA_FS)

    # ── 4. Common 32 Hz raw representation for OPTIONAL CNN-LSTM ─────────────
    from math import gcd

    bvp_clean = butter_filter(bvp_raw, BVP_FS, [0.5, 4.0], btype='band', order=3)

    g = gcd(TARGET_FS, BVP_FS)
    bvp_ds = resample_poly(bvp_clean, TARGET_FS // g, BVP_FS // g)

    g = gcd(TARGET_FS, EDA_FS)
    eda_ds = resample_poly(eda_clean, TARGET_FS // g, EDA_FS // g)

    n_ds = min(len(bvp_ds), len(eda_ds), int(np.floor(duration * TARGET_FS)))
    bvp_ds, eda_ds = bvp_ds[:n_ds], eda_ds[:n_ds]

    # ── 5. Window by common time ─────────────────────────────────────────────
    feats, raws, ys = [], [], []

    for s in range(0, n_ds - WINDOW_LEN + 1, STEP_LEN):
        e  = s + WINDOW_LEN
        t0 = s / TARGET_FS
        t1 = e / TARGET_FS

        # Native 700 Hz labels for this exact time interval.
        la = max(0, int(np.floor(t0 * LABEL_FS)))
        lb = min(len(labels), int(np.ceil(t1 * LABEL_FS)))
        seg = labels[la:lb]
        if len(seg) == 0:
            continue

        vals, counts = np.unique(seg, return_counts=True)
        dominant = int(vals[np.argmax(counts)])
        purity   = float(np.max(counts) / len(seg))

        # Keep only clean baseline or stress windows.
        if dominant not in LABEL_MAP or purity < LABEL_PURITY:
            continue
        y = LABEL_MAP[dominant]

        # Native-rate EDA slice.
        ea = max(0, int(np.floor(t0 * EDA_FS)))
        eb = min(len(eda_clean), int(np.ceil(t1 * EDA_FS)))
        if eb - ea < 3:
            continue
        f_eda = eda_window_features(eda_clean[ea:eb], tonic[ea:eb], phasic[ea:eb], EDA_FS)

        # Pulse intervals whose midpoint falls inside the same window.
        m = (ibi_mid_t >= t0) & (ibi_mid_t < t1)
        f_bvp = pulse_interval_features(ibi_all[m])

        feats.append(f_eda + f_bvp)
        raws.append(np.stack([eda_ds[s:e], bvp_ds[s:e]], axis=1))
        ys.append(y)

    if not feats:
        return None

    return (
        np.asarray(feats, dtype=np.float32),
        np.asarray(raws,  dtype=np.float32),
        np.asarray(ys,    dtype=np.int64),
        np.full(len(ys), sid, dtype=np.int64),
    )


print('WESAD wrist BVP + EDA subject processor ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  BUILD THE BVP + EDA DATASET
# ══════════════════════════════════════════════════════════════════════════════

FEAT_NPZ = os.path.join(OUT_PATH, 'dataset_bvp_eda.npz')

if RUN_PREPROCESS:
    t_start = time.time()
    F, R, Y, G = [], [], [], []

    for sid in SUBJECTS:
        t0  = time.time()
        out = process_subject(sid, WESAD_PATH)
        if out is None:
            print(f'  S{sid}: no usable windows — skipped')
            continue
        f, r, y, g = out
        F.append(f); R.append(r); Y.append(y); G.append(g)
        print(f'  S{sid}: {len(y):5d} windows  '
              f'(baseline={int((y==0).sum()):4d}, stress={int((y==1).sum()):4d})  '
              f'[{time.time()-t0:.1f}s]')

    X_feat = np.concatenate(F).astype(np.float32)
    X_raw  = np.concatenate(R).astype(np.float32)
    y_all  = np.concatenate(Y).astype(np.int64)
    groups = np.concatenate(G).astype(np.int64)

    np.savez_compressed(FEAT_NPZ, X_feat=X_feat, X_raw=X_raw, y=y_all, groups=groups,
                        feature_cols=np.array(FEATURE_COLS))
    print(f'\nSaved → {FEAT_NPZ}   ({time.time()-t_start:.0f}s total)')

else:
    d = np.load(FEAT_NPZ, allow_pickle=True)
    X_feat, X_raw, y_all, groups = d['X_feat'], d['X_raw'], d['y'], d['groups']
    FEATURE_COLS = list(d['feature_cols'])
    print(f'Loaded cached dataset from {FEAT_NPZ}')

print(f'\nX_feat : {X_feat.shape}')
print(f'X_raw  : {X_raw.shape}')
print(f'y      : {y_all.shape}   baseline={int((y_all==0).sum())}, stress={int((y_all==1).sum())}'
      f'   ({100*(y_all==1).mean():.1f}% stress)')
print(f'groups : {len(np.unique(groups))} subjects')

---
# PART A2 · Duplicate removal and per-subject normalization

Absolute EDA level and resting pulse timing differ between people and devices. This section optionally expresses each person's values relative to a calibration block and removes definitional duplicate features.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  A2.1 — Drop definitional duplicates
# ══════════════════════════════════════════════════════════════════════════════
if DROP_DUPLICATE_FEATURES:
    keep = [c for c in FEATURE_COLS if c not in DUPLICATE_FEATURES]
    idx  = [FEATURE_COLS.index(c) for c in keep]
    print(f'Dropping {len(FEATURE_COLS) - len(keep)} definitional duplicates: '
          f'{[c for c in FEATURE_COLS if c in DUPLICATE_FEATURES]}')
    X_feat, FEATURE_COLS = X_feat[:, idx], keep
print(f'Features in use: {len(FEATURE_COLS)}')


# ══════════════════════════════════════════════════════════════════════════════
#  A2.2 — Per-subject normalisation
# ══════════════════════════════════════════════════════════════════════════════
def calibration_index(sid):
    """Boolean mask (within this subject) of the windows used as their reference."""
    m = groups == sid
    n = int(m.sum())
    sel = np.zeros(n, dtype=bool)
    if CALIB_MODE == 'first_k':
        sel[:min(CALIB_WINDOWS, n)] = True          # recording order is preserved
    elif CALIB_MODE == 'baseline_label':
        sel = (y_all[m] == 0)                       # uses the label — upper bound only
    else:
        sel[:] = True
    return sel


if PER_SUBJECT_NORM and CALIB_MODE != 'none':
    Xf_norm = X_feat.astype(np.float64).copy()
    Xr_norm = X_raw.astype(np.float64).copy()
    report  = []

    for sid in np.unique(groups):
        m   = groups == sid
        sel = calibration_index(sid)

        # ── tabular features ──
        block = Xf_norm[m]
        mu = np.nanmean(block[sel], axis=0)
        sd = np.nanstd (block[sel], axis=0)
        sd[~np.isfinite(sd) | (sd < 1e-8)] = 1.0            # constant during calibration
        mu[~np.isfinite(mu)] = 0.0
        Xf_norm[m] = (block - mu) / sd

        # ── raw waveform windows, per channel ──
        rblock = Xr_norm[m]
        rmu = rblock[sel].mean(axis=(0, 1))
        rsd = rblock[sel].std (axis=(0, 1))
        rsd[rsd < 1e-8] = 1.0
        Xr_norm[m] = (rblock - rmu) / rsd

        report.append({'subject': f'S{sid}', 'calib_windows': int(sel.sum()),
                       'calib_stress_frac': float(y_all[m][sel].mean()),
                       'eda_mean_before': float(np.nanmean(X_feat[m][:, FEATURE_COLS.index('eda_mean')]))})

    X_feat = Xf_norm.astype(np.float32)
    X_raw  = Xr_norm.astype(np.float32)

    rep = pd.DataFrame(report)
    print(f"\nPer-subject normalisation applied — mode='{CALIB_MODE}'\n")
    print(rep.round(3).to_string(index=False))

    if CALIB_MODE == 'first_k':
        frac = rep.calib_stress_frac.max()
        print(f'\nMax stress fraction inside any calibration block: {frac:.3f}')
        print('  Should be ~0.00 — WESAD begins with the baseline condition. If it is not,'
              '\n  lower CALIB_WINDOWS or switch to CALIB_MODE = "baseline_label".')

    # ── Did the between-subject spread actually shrink? ──────────────────────
    # Raw and normalised features are in different units, so their SDs cannot be
    # compared directly. Use a scale-free ratio instead:
    #     between-subject SD of the per-subject mean  /  pooled SD of all values
    # This is the fraction of total variance that is "which person is this"
    # rather than "what is happening". Lower is better — it means the feature is
    # carrying signal about state instead of identity.
    raw   = np.load(FEAT_NPZ, allow_pickle=True)
    raw_cols = list(raw['feature_cols'])
    key = [k for k in ['eda_mean', 'bvp_mean_ibi', 'bvp_prv_rmssd', 'eda_phasic_std']
           if k in FEATURE_COLS and k in raw_cols]

    def identity_ratio(vals):
        v = pd.Series(vals).groupby(groups)
        return float(v.mean().std() / (np.nanstd(vals) + 1e-12))

    print('\nBetween-subject SD as a fraction of total SD (lower = less identity leakage):')
    print(f"  {'feature':18s}  {'before':>8s}  {'after':>8s}   change")
    for k in key:
        b = identity_ratio(raw['X_feat'][:, raw_cols.index(k)])
        a = identity_ratio(X_feat[:, FEATURE_COLS.index(k)])
        print(f'  {k:18s}  {b:8.3f}  {a:8.3f}   {100*(1-a/max(b,1e-9)):+6.0f}%')
    print('\nIf these barely move, the failing subjects are not level outliers and the\n'
          'diagnostic cell in Part F will say so — look at raw signal quality instead.')

else:
    print('Per-subject normalisation OFF — using global scaling only.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  SAVE THE FINAL PREPROCESSED REPRESENTATION
# ══════════════════════════════════════════════════════════════════════════════
FINAL_NPZ = os.path.join(OUT_PATH, 'dataset_bvp_eda_normalized.npz')
np.savez_compressed(
    FINAL_NPZ,
    X_feat=X_feat,
    X_raw=X_raw,
    y=y_all,
    groups=groups,
    feature_cols=np.asarray(FEATURE_COLS),
    per_subject_norm=np.asarray(PER_SUBJECT_NORM),
    calib_mode=np.asarray(CALIB_MODE),
    calibration_sec=np.asarray(CALIB_SEC),
)

print(f'Saved final preprocessed dataset: {FINAL_NPZ}')
print(f'X_feat: {X_feat.shape}')
print(f'X_raw : {X_raw.shape} (channels: [EDA, BVP])')
print(f'y     : {y_all.shape}')
print(f'groups: {groups.shape} ({len(np.unique(groups))} subjects)')
print(f'features: {len(FEATURE_COLS)}')
